# Deriving the 2026-09 deposits


Every column added in the September 2026 data audit comes from a public deposit through a computation -- a mean of replicate screens, a log, a moderated contrast. This notebook runs each one from the raw file and shows the check that admitted it. The code is `starplast/deposits.py`; this notebook calls it, so what is shown is what ships.

Raw deposits live under the dataset root, in the taxonomy `<level>/<kind>/<PMID or accession>/`; every folder carries `URLS.txt` and `SHA256SUMS.txt` from the download.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from scipy import stats
ROOT = '/media/carruthers/mnt3/claude/repo/starplast'
DATA = '/media/carruthers/mnt3/claude/toxoplasma_projects/datasets'
import sys; sys.path.insert(0, ROOT)
from starplast import deposits as D
nodes = pd.read_parquet(os.path.join(ROOT, 'starplast', 'data', 'nodes.parquet'))
product = nodes.set_index('gene_id')['product']
ribosomal = product.str.contains('ribosomal protein', case=False, na=False) & ~product.str.contains('mitochondrial|apicoplast|kinase|methyltransferase|acetyltransferase', case=False, na=False)
def me49(ids): return 'TGME49_' + pd.Series(ids, dtype=str).str.extract(r'_(\d{6})')[0]
print(len(nodes), 'genes;', int(ribosomal.sum()), 'cytosolic ribosomal proteins')

8140 genes; 155 cytosolic ribosomal proteins


## 1. In vivo fitness in six tissues (Giuliano et al. 2024, PMID 38977907)

The four tissues already shipped were cited to the 2019 platform paper. If they are this study's composite scores, the supplement must reproduce them value for value. It does, and it carries heart and brain as well.

In [2]:
invivo = D.giuliano_invivo(DATA)
invivo['me49'] = me49(invivo.gene_id).values
whole = invivo[~invivo.gene_id.str.contains(r'\d[A-Z]$')].drop_duplicates('me49').set_index('me49')
rows = []
for c in ['fit_invivo_PE', 'fit_invivo_liver', 'fit_invivo_spleen', 'fit_invivo_lung']:
    j = nodes.set_index('gene_id')[[c]].join(whole[[c]], rsuffix='_supp').dropna()
    rows.append({'column': c, 'genes': len(j), 'max |difference|': (j[c] - j[c + '_supp']).abs().max(),
                 'spearman': stats.spearmanr(j[c], j[c + '_supp']).correlation})
pd.DataFrame(rows)

,column,genes,max |difference|,spearman
0,fit_invivo_PE,7395,4.977e-08,1
1,fit_invivo_liver,7395,4.939e-08,1
2,fit_invivo_spleen,7395,4.878e-08,1
3,fit_invivo_lung,7395,4.99e-08,1


In [3]:
pd.DataFrame({t: invivo[f'fit_invivo_{t}'].describe() for t in ['PE', 'lung', 'heart', 'brain']}).round(2)

,PE,lung,heart,brain
count,8332,8332,8332,8332
mean,-2.89,-5.95,-3.97,-1.79
std,33.83,27.94,6.83,3.6
min,-798.9,-835.7,-79.9,-31.65
25%,-1.53,-3.61,-5.53,-2.77
50%,-0.05,-0.3,-1.5,-0.68
75%,0.11,0.02,-0.07,-0.01
max,515.6,410.6,73.6,38.32


## 2. Serum restriction (Bitew et al. 2025, PMID 41407671)

Two independent genome-wide screens, each in 10% and 1% serum. A screen of fibroblast fitness must find the ribosome essential; the difference between sera must NOT be fibroblast fitness again, or it is no new axis. GRA38 is the paper's gene.

In [4]:
serum = D.serum_restriction(DATA)
serum['me49'] = me49(serum.gene_id).values
s = serum[~serum.gene_id.str.contains(r'\d[A-Z]$')].drop_duplicates('me49').set_index('me49')
s = s.join(nodes.set_index('gene_id')[['fit_invitro_hff']])
rib = ribosomal.reindex(s.index).fillna(False).astype(bool)
pd.DataFrame({c: {'ribosomal median': s.loc[rib, c].median(), 'others median': s.loc[~rib, c].median(),
                  'rho with fit_invitro_hff': stats.spearmanr(s[c], s.fit_invitro_hff, nan_policy='omit').correlation}
              for c in s.columns if c.startswith('fit_') and c != 'fit_invitro_hff'}).T.round(3)

,ribosomal median,others median,rho with fit_invitro_hff
fit_lipid_rich_p8,-6.353,-3.119,0.847
fit_lipid_limited_p8,-6.126,-2.823,0.847
fit_lipid_rich_p4p5,-5.176,-1.397,0.842
fit_lipid_limited_p4p5,-5.253,-1.291,0.844
fit_serum_differential_p8,-0.201,-0.21,0.064
fit_serum_differential_p4p5,0.139,0.034,-0.209


In [5]:
serum.set_index('gene_id').loc[['TGGT1_312420']].T  # GRA38

gene_id,TGGT1_312420
fit_lipid_rich_p8,-3.637
fit_lipid_limited_p8,1.491
fit_lipid_rich_p4p5,1.159
fit_lipid_limited_p4p5,2.223
fit_serum_differential_p8,-5.128
fit_serum_differential_p4p5,-1.064
me49,TGME49_312420


## 3. Translation efficiency, GSE302107

The authors' TE is a linear ratio of footprint RPKM to RNA RPKM; it is shipped as log2 to sit on the scale of the other TE columns. The test is reproducibility, agreement with the earlier studies, and the ribosome being the high-TE class.

In [6]:
te = D.riboseq_302107(DATA).set_index('gene_id')
other = nodes.set_index('gene_id').filter(regex='^te(99395_intracellular|245775_parent_tachy)')
{'replicate rho': stats.spearmanr(te.te302107_tachy_r1, te.te302107_tachy_r2, nan_policy='omit').correlation,
 'rho with GSE99395 mean': stats.spearmanr(te.mean(axis=1), other.filter(like='99395').mean(axis=1).reindex(te.index), nan_policy='omit').correlation,
 'rho with GSE245775 mean': stats.spearmanr(te.mean(axis=1), other.filter(like='245775').mean(axis=1).reindex(te.index), nan_policy='omit').correlation,
 'ribosomal median log2 TE': te.mean(axis=1)[ribosomal.reindex(te.index).fillna(False).astype(bool)].median(),
 'others median log2 TE': te.mean(axis=1)[~ribosomal.reindex(te.index).fillna(False).astype(bool)].median()}

{'replicate rho': np.float64(0.9717565573467376), 'rho with GSE99395 mean': np.float64(0.7487017938696973), 'rho with GSE245775 mean': np.float64(0.7494289059739437), 'ribosomal median log2 TE': np.float64(1.3644291234578687), 'others median log2 TE': np.float64(0.1144645420467956)}

## 4. mRNA decay after actinomycin D, GSE329845

Raw counts, no spike-in: the result is decay relative to the median transcript. The model is intercept + treatment + replicate, moderated as in limma. Replicate agreement is checked on the per-replicate paired log ratios; stable ribosomal mRNAs are the biology check.

In [7]:
decay, fit = D.mrna_decay_329845(DATA, return_fit=True)
norm = fit['normalized']
paired = pd.DataFrame({i: np.log2(norm[f'cWT_ActD_REP{i}'] + 1) - np.log2(norm[f'cWT_vehicle_REP{i}'] + 1) for i in (1, 2, 3)}).loc[decay.gene_id]
print('size factors', {k: round(v, 3) for k, v in fit['size_factors'].items()})
print('prior df', round(fit['df_prior'], 1))
paired.corr(method='spearman').round(3)

size factors {'cWT_vehicle_REP1': 3.142, 'cMUT_vehicle_REP1': 4.156, 'cWT_vehicle_REP2': 3.139, 'cMUT_vehicle_REP2': 2.62, 'cWT_vehicle_REP3': 3.226, 'cMUT_vehicle_REP3': 3.029, 'cWT_ActD_REP1': 0.39, 'cMUT_ActD_REP1': 0.411, 'cWT_ActD_REP2': 0.292, 'cMUT_ActD_REP2': 0.245, 'cWT_ActD_REP3': 0.299, 'cMUT_ActD_REP3': 0.309}
prior df 4.4


,1,2,3
1,1,0.966,0.963
2,0.966,1,0.961
3,0.963,0.961,1


In [8]:
d = decay.assign(me49=me49(decay.gene_id).values).set_index('me49')['mrna_log2_remaining_4h_actinomycin']
rib = ribosomal.reindex(d.index).fillna(False).astype(bool)
old = nodes.set_index('gene_id')['mrna_remaining_5h_actinomycin'].reindex(d.index)
{'genes': len(d), 'ribosomal median': d[rib].median(), 'others median': d[~rib].median(),
 'Mann-Whitney p': stats.mannwhitneyu(d[rib], d[~rib]).pvalue,
 'rho with the shipped 412-gene column': stats.spearmanr(d, old, nan_policy='omit').correlation,
 'shared genes': int(old.notna().sum())}

{'genes': 6406, 'ribosomal median': np.float64(1.4255177589843855), 'others median': np.float64(-0.1696065040180186), 'Mann-Whitney p': np.float64(3.2732681637830885e-14), 'rho with the shipped 412-gene column': np.float64(-0.02104841441400259), 'shared genes': 298}

## 5. Host responses and a baseline macrophage

Host tables are keyed by reviewed UniProt accession through Ensembl ids (`reference/uniprot`), never by symbol. Each contrast is infected against uninfected with the design's blocks; the checks are genes whose behaviour is not in doubt.

In [9]:
hff = D.hff_tg_infection(DATA).drop_duplicates('host_name').set_index('host_name')
print(int((hff.hff_tg_infection_padj < 0.05).sum()), 'genes at padj < 0.05 of', len(hff))
hff.reindex(['CXCL8', 'IL6', 'CXCL10', 'ISG15', 'EGR1', 'GAPDH', 'ACTB']).round(3)

315 genes at padj < 0.05 of 10630


,hff_tg_infection_log2fc,hff_tg_infection_padj,host_id
host_name,,,
CXCL8,5.353,0,P10145
IL6,2.666,0,P05231
CXCL10,2.262,0.004,P02778
ISG15,2.134,0.004,P05161
EGR1,1.236,0.029,P18146
GAPDH,-0.025,0.871,P04406
ACTB,-0.028,0.887,P60709


In [10]:
bmdm = D.bmdm_baseline(DATA).drop_duplicates('host_name').set_index('host_name')
bmdm['rank'] = bmdm.bmdm_tpm.rank(ascending=False).astype(int)
bmdm.reindex(['Lyz2', 'Cd68', 'Csf1r', 'Emr1', 'Itgam', 'Alb', 'Apoa1']).round(1)

,bmdm_tpm,host_id,rank
host_name,,,
Lyz2,2.75e+04,P08905,1
Cd68,2791,P31996,39
Csf1r,1276,P09581,123
Emr1,903.2,Q61549,164
Itgam,NaN,NaN,NaN
Alb,0,P07724,1.414e+04
Apoa1,0,Q00623,1.414e+04


In [11]:
hep = D.hepatocyte_pf_infection(DATA).drop_duplicates('host_name').set_index('host_name')
print(int((hep.hepatocyte_pf_infection_padj < 0.05).sum()), 'genes at padj < 0.05 of', len(hep))
hep.sort_values('hepatocyte_pf_infection_log2fc', ascending=False).head(10).round(3)

25 genes at padj < 0.05 of 12550


,hepatocyte_pf_infection_log2fc,hepatocyte_pf_infection_padj,host_id
host_name,,,
IFI44L,4.481,0.077,Q53G44
CXCL10,2.674,0.076,P02778
RSAD2,2.336,0.083,Q8WXG1
CXCL11,2.284,0.025,O14625
MX2,2.264,0.092,P20592
CTHRC1,2.187,0.006,Q96CG8
SFRP1,2.178,0.134,Q8N474
OASL,2.171,0.006,Q15646
HGF,2.021,0.116,P14210


## 6. Write the tables

Each derivation is written to `starplast/data/deposit_<key>.tsv`. `scripts/add_deposits.py` merges them into the node and host tables, refusing any merge that would lose a value.

In [12]:
written = D.derive_all(DATA, ROOT, log=print)
{k: v.shape for k, v in written.items()}

deposit crispr_invivo_composite: 8,332 rows x 6 columns
deposit crispr_serum_restriction: 8,158 rows x 6 columns
deposit gse302107_riboseq: 5,992 rows x 2 columns
deposit mrna_decay_gse329845: 6,406 rows x 1 columns
deposit host_hff_tg_infection: 10,631 rows x 3 columns
deposit host_bmdm_baseline: 15,437 rows x 2 columns
deposit host_hepatocyte_pf_infection: 12,550 rows x 3 columns


{'crispr_invivo_composite': (8332, 7), 'crispr_serum_restriction': (8158, 7), 'gse302107_riboseq': (5992, 3), 'mrna_decay_gse329845': (6406, 2), 'host_hff_tg_infection': (10631, 4), 'host_bmdm_baseline': (15437, 3), 'host_hepatocyte_pf_infection': (12550, 4)}